# Offline Dataloader Caching Strategy

Este notebook executa a estruturação de cache offline para os crops do dataset LUNA16. 
Ao usar `get_ct(...)` direto no loop de treinamento, o PyTorch era forçado a carregar centenas de MegaBytes do arquivo `.mhd` repetidamente a cada iteração de época.

A solução: Iterar sobre os candidatos antes do treinamento, recortar o bloco `32x48x48` contendo (ou não) o nódulo, e salvá-lo no disco como minúsculos arquivos `torch.Tensor` (`.pt`). O tamanho cai de Gigabytes de leitura da memória por *batch* para apenas alguns Kilobytes.

## 1. Importações

In [1]:
import sys
from pathlib import Path

# Adicionar src/ ao python path para conseguirmos importar nossos módulos
sys.path.insert(0, str(Path().resolve().parent / "src"))

import torch
from tqdm.notebook import tqdm
from luna_data import load_candidates, get_ct, get_cache_path, CACHE_DIR

## 2. Inicializar a rotina de Geração do Cache

O bloco abaixo varrerá todos os candidatos apontados pelo `annotations.csv` / `candidates.csv`, acessando suas coordenas XYZ originais. Em seguida, gera os Crops de tensores pré-estabelecidos e os escreve dentro da pasta `data/luna/cache/`.

In [2]:
def build_cache():
    print(f"Diretório de cache: {CACHE_DIR}")
    candidates = load_candidates()
    print(f"Total de candidatos encontrados na base: {len(candidates)}")
    
    # CRÍTICO: Precisamos ordenar pelas IDs das tomografias para garantir 
    # que todas as amostras de uma tomografia sejam processadas de uma só vez.
    # O get_ct usa @functools.lru_cache(1), ou seja, isso reduzirá o tempo
    # para construir a base de algumas centenas de HORAS para meros MINUTOS!!!
    candidates_sorted = sorted(candidates, key=lambda c: c.series_uid)
    
    # Vamos salvar os crops de tensores de cada candidato em ordem contígua
    for cand in tqdm(candidates_sorted, desc="Gerando Cache dos Crops"):
        cache_path = get_cache_path(cand.series_uid, cand.center_xyz)
        
        # Se o cache já foi construído por uma sessão anterior, simplesmente pular
        if cache_path.exists():
            continue
            
        # Carrega a varredura primária, aplica extração
        ct = get_ct(cand.series_uid)
        crop_a, center_irc = ct.extract_crop(cand.center_xyz)
        
        # Converte a extração e grava no disco local sob um hash MD5 determinístico
        crop_t = torch.from_numpy(crop_a).to(torch.float32).unsqueeze(0).clone()
        
        # Salvando na pasta
        torch.save((crop_t, center_irc), cache_path)
        
    print("\nConstrução da base de processamento Offline Caching finalizada com sucesso!")

## 3. Executar o Cacheamento
> **AVISO:** Este processo pode levar de alguns minutos até algumas horas a depender do IO (Leitura) máximo do seu disco. Como é feito apenas uma única vez para toda a vida do projeto, é extremamente recomendável executá-lo.
Sente-se, vá tomar um café e aguarde a barra de preenchimento chegar a 100%!

In [3]:
# Inicia o processo pesado
build_cache()

Diretório de cache: D:\Deep-learning - Deteccao de nodulo\data\luna\cache
Total de candidatos encontrados na base: 551065


Gerando Cache dos Crops:   0%|          | 0/551065 [00:00<?, ?it/s]


Construção da base de processamento Offline Caching finalizada com sucesso!
